# Лабораторная работа: Деревья и ансамбли в sklearn по CRISP-DM

Датасет UCI Bank Marketing.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_auc_score, precision_score, recall_score, f1_score, balanced_accuracy_score
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
DATA_PATH = Path("data/bank-full.csv")
df_bank = pd.read_csv(DATA_PATH, sep=";")
print("Размер:", df_bank.shape)
print("Баланс y:\n", df_bank["y"].value_counts(normalize=True))


## Фазы 1–2. Business Understanding & Data Understanding

Цель — предсказать согласие клиента на депозит до совершения телефонного звонка.

`duration` исключается из основной модели, поскольку становится известен только после звонка и является target leakage.

Значение `pdays=-1` означает отсутствие предыдущего контакта. Оно преобразуется в индикатор `was_contacted` и очищается от специального кода.

In [ ]:
y = (df_bank["y"] == "yes").astype(int).values
X = df_bank.drop(columns=["y", "duration"])

numeric_features = ["age","balance","day","campaign","pdays","previous"]
categorical_features = ["job","marital","education","default","housing","loan","contact","month","poutcome"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)

def process_pdays(df_sub):
    out = df_sub.copy()
    p = out["pdays"]
    out["was_contacted"] = (p != -1).astype(float)
    out["pdays_clean"] = np.where(p == -1, 0.0, p)
    return out.drop(columns=["pdays"])

pdays_transformer = FunctionTransformer(process_pdays)

num_pipe = Pipeline([
    ("pdays_proc", pdays_transformer),
    ("imputer", SimpleImputer(strategy="median"))
])
cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])
preprocessor = ColumnTransformer([
    ("num", num_pipe, numeric_features),
    ("cat", cat_pipe, categorical_features)
])

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)


## Фазы 3–4. Data Preparation, Pipelines & Baselines

In [ ]:
base_models = {
    "Prior Dummy": DummyClassifier(strategy="prior"),
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "Bagging": BaggingClassifier(random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE)
}

baseline_cv_scores = {}
for name, estimator in base_models.items():
    pipe = Pipeline([("prep", preprocessor), ("model", estimator)])
    scores = []
    for tr_idx, val_idx in cv.split(X_train, y_train):
        pipe.fit(X_train.iloc[tr_idx], y_train[tr_idx])
        probs = pipe.predict_proba(X_train.iloc[val_idx])[:, 1]
        scores.append(average_precision_score(y_train[val_idx], probs))
    baseline_cv_scores[name] = np.mean(scores)

df_baseline = pd.DataFrame(
    list(baseline_cv_scores.items()),
    columns=["Model", "CV PR-AUC (Average Precision)"]
)
display(df_baseline)


## Фаза 5. Hyperparameter Tuning

In [ ]:
dt_pipe = Pipeline([
    ("prep", preprocessor),
    ("model", DecisionTreeClassifier(random_state=RANDOM_STATE))
])
dt_grid = {
    "model__max_depth": [3,5,8,12],
    "model__min_samples_leaf": [5,15,30],
    "model__criterion": ["gini","entropy"]
}
dt_search = GridSearchCV(
    dt_pipe, dt_grid, scoring="average_precision", cv=cv,
    n_jobs=-1, refit=True
).fit(X_train, y_train)

bag_pipe = Pipeline([
    ("prep", preprocessor),
    ("model", BaggingClassifier(random_state=RANDOM_STATE))
])
bag_dist = {
    "model__n_estimators": [20,50,80],
    "model__max_samples": [0.5,0.8,1.0],
    "model__max_features": [0.5,0.8,1.0]
}
bag_search = RandomizedSearchCV(
    bag_pipe, bag_dist, n_iter=6, scoring="average_precision",
    cv=cv, random_state=RANDOM_STATE, n_jobs=-1, refit=True
).fit(X_train, y_train)

rf_pipe = Pipeline([
    ("prep", preprocessor),
    ("model", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1))
])
rf_dist = {
    "model__n_estimators": [50,100,150],
    "model__max_depth": [6,10,15,None],
    "model__min_samples_leaf": [2,5,10],
    "model__max_features": ["sqrt",0.5]
}
rf_search = RandomizedSearchCV(
    rf_pipe, rf_dist, n_iter=6, scoring="average_precision",
    cv=cv, random_state=RANDOM_STATE, n_jobs=-1, refit=True
).fit(X_train, y_train)

gb_pipe = Pipeline([
    ("prep", preprocessor),
    ("model", GradientBoostingClassifier(random_state=RANDOM_STATE))
])
gb_dist = {
    "model__n_estimators": [50,100,150],
    "model__learning_rate": [0.03,0.08,0.15],
    "model__max_depth": [3,4,5],
    "model__subsample": [0.8,1.0]
}
gb_search = RandomizedSearchCV(
    gb_pipe, gb_dist, n_iter=6, scoring="average_precision",
    cv=cv, random_state=RANDOM_STATE, n_jobs=-1, refit=True
).fit(X_train, y_train)

for name, search in {
    "Decision Tree": dt_search,
    "Bagging": bag_search,
    "Random Forest": rf_search,
    "Gradient Boosting": gb_search
}.items():
    print(name, "PR-AUC =", round(search.best_score_, 4))
    print("Best params:", search.best_params_)


## Фаза 6. Evaluation on Unseen Test Set

In [ ]:
tuned_models = {
    "Decision Tree": dt_search.best_estimator_,
    "Bagging": bag_search.best_estimator_,
    "Random Forest": rf_search.best_estimator_,
    "Gradient Boosting": gb_search.best_estimator_
}

test_metrics = []
for name, model in tuned_models.items():
    probs = model.predict_proba(X_test)[:,1]
    preds = model.predict(X_test)
    test_metrics.append({
        "Model": name,
        "PR-AUC": average_precision_score(y_test, probs),
        "ROC-AUC": roc_auc_score(y_test, probs),
        "Precision": precision_score(y_test, preds, zero_division=0),
        "Recall": recall_score(y_test, preds, zero_division=0),
        "F1-score": f1_score(y_test, preds, zero_division=0),
        "Balanced Accuracy": balanced_accuracy_score(y_test, preds)
    })

df_test_eval = pd.DataFrame(test_metrics)
display(df_test_eval)


### Обоснование метрики

Положительный класс составляет около 11.7%. Поэтому основной метрикой выбора является PR-AUC (Average Precision), которая лучше отражает качество ранжирования редкого положительного класса.

## Фаза 7. Feature Importance & Permutation Importance

In [ ]:
perm_res = permutation_importance(
    rf_search.best_estimator_,
    X_test, y_test,
    scoring="average_precision",
    n_repeats=5,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

df_perm = pd.DataFrame({
    "Feature": X_test.columns,
    "Perm_Mean": perm_res.importances_mean,
    "Perm_Std": perm_res.importances_std
}).sort_values("Perm_Mean", ascending=False)

display(df_perm.head(10))

top_perm = df_perm.head(8)
plt.figure(figsize=(10,5))
plt.barh(top_perm["Feature"], top_perm["Perm_Mean"], xerr=top_perm["Perm_Std"])
plt.gca().invert_yaxis()
plt.xlabel("Падение PR-AUC при перемешивании")
plt.title("Permutation Feature Importance")
plt.grid(True, alpha=0.3)
plt.show()


## Фаза 8. Контролируемый эксперимент с утечкой

In [ ]:
X_train_leak = df_bank.iloc[X_train.index].drop(columns=["y"])
leak_num_features = numeric_features + ["duration"]

leak_preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("pdays", pdays_transformer),
        ("imp", SimpleImputer(strategy="median"))
    ]), leak_num_features),
    ("cat", cat_pipe, categorical_features)
])

leak_pipe = Pipeline([
    ("prep", leak_preprocessor),
    ("model", DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE))
])

leak_scores = []
for tr_idx, val_idx in cv.split(X_train_leak, y_train):
    leak_pipe.fit(X_train_leak.iloc[tr_idx], y_train[tr_idx])
    probs = leak_pipe.predict_proba(X_train_leak.iloc[val_idx])[:,1]
    leak_scores.append(average_precision_score(y_train[val_idx], probs))

print(f"PR-AUC без duration: {dt_search.best_score_:.4f}")
print(f"PR-AUC с duration: {np.mean(leak_scores):.4f}")


## Модельная карточка

1. Бизнес-задача: приоритизация клиентов для маркетинговой кампании.
2. Источник: UCI Bank Marketing, 45 211 записей.
3. `duration` исключён из основной модели из-за target leakage.
4. `pdays=-1` преобразован в индикатор предыдущего контакта.
5. Train/Test = 80/20 со стратификацией.
6. Гиперпараметры выбираются через 3-Fold Stratified CV.
7. Основная метрика — PR-AUC.
8. Permutation Importance рассчитывается на тестовой выборке.
9. Модель предназначена для приоритизации маркетинговых списков и не должна использоваться для автоматизированных решений о кредитовании или обслуживании.
